# Basket features

What is in the order: bracketing (same article in several sizes or colors), discount depth, and basket mix. Bracketing is ordering variants intending to keep one, so it is the clearest return signal available at checkout.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from src.data import INTERIM, load_orders, save_features

## Lines

**Line-level derived columns.** `discount` is guarded by `np.where(rrp > 0, ...)` because 318 lines carry `rrp = 0`; those become NaN, which LightGBM handles natively. Filling them with 0 would assert *no discount*, a different and false claim.

In [2]:
lines = load_orders("orders_train")
lines["line_value"] = lines.price * lines.quantity
lines["discount"] = np.where(lines.rrp > 0, 1 - lines.price / lines.rrp, np.nan)
lines.shape

(2325165, 17)

## Bracketing

**Bracketing, via a two-stage groupby.** Stage one counts distinct sizes and colours of *each article within each order*; stage two takes the max across articles. `multi_size = 1` means the same item was ordered in two sizes — nobody wears both, so one is coming back. This is the single strongest checkout-time signal in the data.

In [3]:
per_article = lines.groupby(["orderID", "articleID"]).agg(
    sizes=("sizeCode", "nunique"), colors=("colorCode", "nunique"), units=("quantity", "sum"))

brack = per_article.groupby("orderID").agg(
    max_sizes_per_article=("sizes", "max"),
    max_colors_per_article=("colors", "max"),
    max_units_per_article=("units", "max"),
    n_articles=("sizes", "size"))
brack["multi_size"] = (brack.max_sizes_per_article > 1).astype(int)
brack["multi_color"] = (brack.max_colors_per_article > 1).astype(int)
brack.head()

,max_sizes_per_article,max_colors_per_article,max_units_per_article,n_articles,multi_size,multi_color
orderID,,,,,,
a1000001,1,1,1,2,0,0
a1000002,1,1,1,2,0,0
a1000003,1,2,2,4,0,1
a1000004,1,1,1,1,0,0
a1000005,2,1,2,2,1,0


## Mix and price

**Basket composition and price shape.** Category spread, discount depth, price dispersion, and voucher use. `has_voucher` compares the ID against the string `"0"` because the raw column encodes 'no voucher' as a literal zero.

In [4]:
mix = lines.groupby("orderID").agg(
    n_groups=("productGroup", "nunique"),
    mean_discount=("discount", "mean"),
    max_discount=("discount", "max"),
    mean_price=("price", "mean"),
    max_price=("price", "max"),
    min_price=("price", "min"),
    mean_rrp=("rrp", "mean"),
    order_value=("line_value", "sum"),
    voucher=("voucherAmount", "max"),
    has_voucher=("voucherID", lambda s: int((s.astype(str) != "0").any())))
mix["price_spread"] = mix.max_price - mix.min_price
mix["voucher_share"] = np.where(mix.order_value > 0, mix.voucher / mix.order_value, 0)
mix = mix.drop(columns=["order_value", "voucher"])
mix.head()

,n_groups,mean_discount,max_discount,mean_price,max_price,min_price,mean_rrp,has_voucher,price_spread,voucher_share
orderID,,,,,,,,,,
a1000001,1,0.583215,0.666556,15.000000,20.00,10.00,34.990000,0,10.00,0.0
a1000002,1,0.149930,0.299860,42.495000,49.99,35.00,49.990000,0,14.99,0.0
a1000003,2,0.688808,1.000000,12.000000,25.00,0.00,42.390000,0,25.00,0.0
a1000004,1,0.000000,0.000000,89.990000,89.99,89.99,89.990000,0,0.00,0.0
a1000005,1,0.652672,0.666556,11.666667,15.00,10.00,33.323333,0,5.00,0.0


## Save

**Join both halves and persist.** Bracketing counts and basket composition merge on `orderID` into a single `basket` block. `save_features` scrubs column names, downcasts to float32 and writes parquet — the same on-disk contract every feature block follows, so notebook 07 can merge them blindly.

In [5]:
basket = brack.join(mix).reset_index()
save_features(basket, "basket", keys=("orderID",))
basket.shape

(738698, 17)